# 01 · Demostración de la cámara de un píxel (SPC)

Este notebook muestra, sobre **una vista de Truck**, las tres etapas 2D del proyecto:

1. **Patrones** de medición (Hadamard ordenado, binario aleatorio y DCT en zigzag).
2. **Medición** `y = Φx`: cada medición es el producto interno de la imagen con un patrón,
   lo que registraría el único fotodetector de la cámara.
3. **Reconstrucción** directa (ceros en lo no medido) frente a variación total (TV) con FISTA,
   al 5, 10, 25 y 50 % de mediciones, con el PSNR de cada panel.

Todo ocurre en la **rejilla del single pixel** (256 × 512 por defecto). Aquí el PSNR se calcula
contra la imagen ya remuestreada a la rejilla, así que mide solo la pérdida debida al SPC.
En `src/make_dataset.py`, en cambio, la comparación se hace a resolución completa contra la
imagen original, y por eso incluye también la pérdida por remuestreo.

In [ ]:
import sys
from pathlib import Path

# Raíz del repositorio (el notebook vive en notebooks/)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.patterns import patron_2d, orden_medicion, indices_medidos
from src.spc import OperadorSPC
from src.recon import reconstruir_directa, reconstruir_tv
from src.make_dataset import cargar_imagen, remuestrear, seleccionar_vistas
from src.metrics import calcular_metricas

device = "cuda" if torch.cuda.is_available() else "cpu"
REJILLA = (256, 512)          # alto x ancho (potencias de 2, requisito de Hadamard)
RAZONES = [0.05, 0.10, 0.25, 0.50]
SEMILLA = 449
LAM_TV = 0.003                # peso de la TV (mismo valor que en make_dataset.py)
print("Dispositivo:", device)

## 1. Vista de ejemplo

Se usa la primera vista de **prueba** del experimento (las mismas 24 vistas equiespaciadas que
usa `make_dataset.py`). La imagen original (979 × 546) se reduce a la rejilla de 256 × 512
promediando por área.

In [ ]:
nombres = sorted(p.name for p in (RAIZ / "data/truck/images").glob("*.jpg"))
entrenamiento, prueba = seleccionar_vistas(nombres, 24, 3)
vista = prueba[0]
x_original = cargar_imagen(RAIZ / "data/truck/images" / vista).to(device)  # (3, 546, 979)
x = remuestrear(x_original, REJILLA)                                       # (3, 256, 512)

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
ax[0].imshow(x_original.permute(1, 2, 0).cpu()); ax[0].set_title(f"Original {vista} {tuple(x_original.shape[1:])}")
ax[1].imshow(x.permute(1, 2, 0).cpu()); ax[1].set_title(f"En la rejilla SPC {REJILLA}")
for a in ax: a.axis("off")
plt.tight_layout()

## 2. Patrones de medición

Cada fila de Φ es un patrón que el modulador espacial (DMD) mostraría frente a la escena.

- **Hadamard ordenado**: patrones ±1 ordenados por secuencia (número de cambios de signo),
  de bajas a altas frecuencias. Se miden los primeros.
- **Binario aleatorio**: Hadamard aplicado a los píxeles permutados con la semilla 449, y
  coeficientes elegidos al azar (siempre incluye el patrón constante). Los patrones son ±1, pero
  sin estructura espacial.
- **DCT (zigzag)**: cosenos, no binarios; referencia de compresión tipo JPEG.

Se muestran los patrones número 1, 2, 3, 10, 100 y 1000 en el orden de medición.

In [ ]:
posiciones = [1, 2, 3, 10, 100, 1000]
fig, ax = plt.subplots(3, len(posiciones), figsize=(16, 5))
for fila, patron in enumerate(["hadamard", "random", "dct"]):
    lista = orden_medicion(patron, REJILLA, SEMILLA)   # orden en que se mide
    for col, k in enumerate(posiciones):
        P = patron_2d(patron, REJILLA, lista[k - 1], SEMILLA)
        ax[fila, col].imshow(P, cmap="gray")
        ax[fila, col].set_title(f"{patron} n.º {k}", fontsize=9)
        ax[fila, col].axis("off")
plt.tight_layout()

## 3. La medición

Para una razón `r`, el single pixel registra `K = r · 256 · 512` números por canal de color
(el vector `y`). Abajo se muestra, para Hadamard y DCT, el mapa de **todos** los coeficientes
(en escala logarítmica) y en blanco los que se miden al 10 %. La energía de una imagen natural
se concentra en las bajas frecuencias (arriba a la izquierda), y por eso el orden de bajas a
altas captura la mayor parte con pocas mediciones.

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(15, 5))
for col, patron in enumerate(["hadamard", "random", "dct"]):
    op_total = OperadorSPC(patron, REJILLA, 1.0, SEMILLA, device)
    C = op_total.transformada(x)[1].abs().cpu().numpy()          # canal verde
    mascara = np.zeros(REJILLA[0] * REJILLA[1])
    mascara[indices_medidos(patron, REJILLA, 0.10, SEMILLA)] = 1
    ax[0, col].imshow(np.log10(C + 1e-4), cmap="magma"); ax[0, col].set_title(f"{patron}: log10 |coeficientes|")
    ax[1, col].imshow(mascara.reshape(REJILLA), cmap="gray"); ax[1, col].set_title(f"{patron}: medidos al 10 %")
    for a in ax[:, col]: a.axis("off")
plt.tight_layout()

op = OperadorSPC("hadamard", REJILLA, 0.10, SEMILLA, device)
y = op.medir(x)
print(f"Al 10 %: y tiene forma {tuple(y.shape)} = 3 canales x {op.num_mediciones} mediciones "
      f"(de {REJILLA[0] * REJILLA[1]} píxeles por canal)")

## 4. Reconstrucción: directa frente a TV

- **Directa**: `x = Φᵀy`, con ceros en lo no medido.
- **TV (FISTA)**: minimiza `½‖Φx − y‖² + λ·TV(x)`, con λ = 0.003 y 100 iteraciones.

Se reconstruyen Hadamard y aleatorio a las cuatro razones. El título de cada panel indica el
PSNR contra la imagen en la rejilla (sin SPC). Con el patrón aleatorio, la reconstrucción
directa es muy mala: las mediciones dispersas no forman una aproximación de bajas frecuencias,
y la TV debe recuperar la estructura.

In [ ]:
filas = [("hadamard", "directa"), ("hadamard", "tv"), ("random", "directa"), ("random", "tv")]
fig, ax = plt.subplots(len(filas), len(RAZONES), figsize=(18, 10))
psnr = {}
for i, (patron, metodo) in enumerate(filas):
    for j, r in enumerate(RAZONES):
        op = OperadorSPC(patron, REJILLA, r, SEMILLA, device)
        y = op.medir(x)
        if metodo == "directa":
            x_rec = reconstruir_directa(y, op)
        else:
            x_rec = reconstruir_tv(y, op, lam=LAM_TV, iteraciones=100)
        x_rec = x_rec.clamp(0, 1)
        m = calcular_metricas(x_rec, x, device)
        psnr[(patron, metodo, r)] = m["psnr"]
        ax[i, j].imshow(x_rec.permute(1, 2, 0).cpu())
        ax[i, j].set_title(f"{patron} · {metodo} · {int(r*100)} %\nPSNR {m['psnr']:.2f} dB", fontsize=10)
        ax[i, j].axis("off")
plt.tight_layout()

### Resumen numérico de esta vista

PSNR (dB) contra la imagen en la rejilla. Son valores de **una sola vista**; los promedios
sobre las 24 vistas, a resolución completa, están en `results/metrics_2d.csv` (notebook 03).

In [ ]:
import pandas as pd
tabla = pd.Series(psnr).unstack(level=2)
tabla.columns = [f"{int(c*100)} %" for c in tabla.columns]
tabla.round(2)